# Part 2: Text Analysis — LSA + OpenAI gpt-4o-mini

**Course:** CSIS 4260 — Douglas College  
**Student:** Desmond Chua  

In this notebook, we apply two text summarization algorithms to the 100 healthcare IT posts scraped in Part 1.

**Algorithm 1 — LSA (Latent Semantic Analysis):**  
An extractive summarization method that runs locally. It uses linear algebra to identify the most important sentences in each post and pulls them out verbatim. No API key required — it's free and fast.

**Algorithm 2 — OpenAI gpt-4o-mini:**  
An abstractive summarization method that uses OpenAI's API. It generates new sentences that capture the meaning of the original text, and can be prompted to focus on healthcare-specific context. Requires an API key.

By comparing both approaches, we can see the trade-offs between extractive (LSA) and abstractive (GPT) summarization on real healthcare forum data.

In [1]:
import pandas as pd

# Load the scraped data from Part 1
# We use ../ because this notebook runs from inside the notebooks/ folder
df = pd.read_csv('../data/scraped_healthcare_posts.csv')

# Check the shape — we expect 100 rows and 3 columns (title, post_url, content)
print(f"DataFrame shape: {df.shape}")

# Check for missing values — NLP functions will break on NaN inputs
print(f"\nMissing values per column:")
print(df.isnull().sum())

# Preview the first 3 rows to make sure the data looks right
print("\nFirst 3 posts:")
print(df.head(3))

print("\nData loaded successfully")

DataFrame shape: (100, 3)

Missing values per column:
title       0
post_url    0
content     2
dtype: int64

First 3 posts:
                                               title  \
0                 "I want to be an Epic analyst" FAQ   
1  Anyone here switch careers without a degree in...   
2  I built a browser-based ambient scribe that ke...   

                                            post_url  \
0  https://www.reddit.com/r/healthIT/comments/1hl...   
1  https://www.reddit.com/r/healthIT/comments/1rx...   
2  https://www.reddit.com/r/healthIT/comments/1rw...   

                                             content  
0  **I'm a [job] and thinking of becoming an Epic...  
1  I'm currently on the clinical side and have be...  
2  For a bit of an experiment, I put together a s...  

Data loaded successfully


## Algorithm 1: LSA Summarizer

**LSA (Latent Semantic Analysis)** is an extractive summarization method — it picks the most important sentences directly from the original text rather than generating new ones.

**How it works in plain English:**
1. It breaks the text into individual sentences
2. It builds a matrix of which words appear in which sentences
3. It uses a linear algebra technique called SVD (Singular Value Decomposition) to figure out which sentences capture the most meaning
4. It returns the top-ranked sentences as the summary

**Why LSA for this project:**
- Free and runs locally — no API key or internet needed
- Great for factual content like forum posts where the original wording matters
- Fast enough to process 100 posts in seconds

**Limitation:** Because it extracts whole sentences, the summaries can sound choppy or disconnected — especially when sentences are pulled from different parts of a long post. We'll compare this with GPT's smoother abstractive summaries later.

In [2]:
# Fill the 2 missing content values with empty string
# so LSA doesn't crash on NaN values
df['content'] = df['content'].fillna("")
print(f"Missing content after fix: {df['content'].isna().sum()}")

Missing content after fix: 0


In [3]:
from sumy.parsers.plaintext import PlaintextParser
from sumy.nlp.tokenizers import Tokenizer
from sumy.summarizers.lsa import LsaSummarizer
import nltk

# Download the tokenizer data that sumy needs to split text into sentences
# Python 3.13 specifically requires punkt_tab or the tokenizer will throw
# a LookupError — we download both to be safe across Python versions
nltk.download('punkt')
nltk.download('punkt_tab')


def summarize_lsa(text, num_sentences=2):
    """
    Summarize a piece of text using LSA (extractive summarization).
    Returns a plain string summary with no line breaks.
    """
    try:
        # If the text is too short, there's nothing to summarize
        # Just return the first 300 characters as-is
        if len(text.split()) < 50:
            return text[:300]

        # Parse the text into sentences so LSA can rank them
        parser = PlaintextParser.from_string(text, Tokenizer("english"))

        # Create the LSA summarizer and run it
        summarizer = LsaSummarizer()
        summary_sentences = summarizer(parser.document, num_sentences)

        # Join the top sentences into a single string with no line breaks
        summary = " ".join(str(sentence) for sentence in summary_sentences)

        # If the summarizer returned nothing, fall back to first 300 chars
        if not summary.strip():
            return text[:300]

        return summary

    except Exception as e:
        # If anything goes wrong, return first 300 chars as a safe fallback
        print(f"  LSA error: {e}")
        return text[:300]


# --- Test LSA on the first 5 posts to make sure it works ---
print("Testing LSA on first 5 posts:\n")

for i in range(5):
    title = df.iloc[i]['title']
    content = df.iloc[i]['content']

    # Handle missing content — use empty string if NaN
    if pd.isna(content):
        content = ""

    summary = summarize_lsa(content)
    print(f"Post {i+1}: {title}")
    print(f"Summary: {summary[:200]}...\n")

print("LSA test complete")

c:\Users\user\OneDrive\Desmond_New\healthcare-nlp-analysis\venv\Lib\site-packages\requests\__init__.py:113: RequestsDependencyWarning: urllib3 (2.6.3) or chardet (7.2.0)/charset_normalizer (3.4.6) doesn't match a supported version!
  warnings.warn(
[nltk_data] Downloading package punkt to
[nltk_data]     C:\Users\user\AppData\Roaming\nltk_data...
[nltk_data]   Package punkt is already up-to-date!
[nltk_data] Downloading package punkt_tab to
[nltk_data]     C:\Users\user\AppData\Roaming\nltk_data...
[nltk_data]   Package punkt_tab is already up-to-date!


Testing LSA on first 5 posts:

Post 1: "I want to be an Epic analyst" FAQ
Summary: Alternatively, keep your ear out for health systems that are transitioning to Epic and apply like crazy at those. You should probably pick something else, given that most entry-level Epic jobs want ex...

Post 2: Anyone here switch careers without a degree in informatics?
Summary: I'm currently on the clinical side and have been looking at ways to break into Health IT for a while now. I got a comptia cert for a couple 100 $s, previous IT-adjacent experience, and had experience ...

Post 3: I built a browser-based ambient scribe that keeps all data on the device (open source)
Summary: * depends on Chrome-specific features * requires fairly modern hardware for on-device models * speech recognition behaviour is browser-dependent * not something you’d use in a real clinical setting (p...

Post 4: How to generate a list of patients in EPIC based on ICD codes for research?
Summary: I’m a student who is trying 

In [4]:
from tqdm import tqdm

# --- Run LSA summarization on all 100 posts ---
print("Running LSA on all 100 posts...\n")

lsa_summaries = []

for i in tqdm(range(len(df)), desc="LSA Summarization"):
    content = df.iloc[i]['content']

    # Handle missing content — use empty string if NaN
    if pd.isna(content):
        content = ""

    # Generate the LSA summary for this post
    summary = summarize_lsa(content)

    # Make sure summary is a clean string with no line breaks
    summary = summary.replace("\n", " ").strip()

    # Final safety check — if summary is empty, use first 300 chars
    if not summary:
        summary = content[:300]

    lsa_summaries.append(summary)

# Store the summaries in a new column
df['lsa_summary'] = lsa_summaries

print("\nLSA complete — sample output:")
print(df[['title', 'lsa_summary']].head(3))

Running LSA on all 100 posts...



LSA Summarization: 100%|██████████| 100/100 [00:12<00:00,  8.02it/s]



LSA complete — sample output:
                                               title  \
0                 "I want to be an Epic analyst" FAQ   
1  Anyone here switch careers without a degree in...   
2  I built a browser-based ambient scribe that ke...   

                                         lsa_summary  
0  Alternatively, keep your ear out for health sy...  
1  I'm currently on the clinical side and have be...  
2  * depends on Chrome-specific features * requir...  


## LSA Results Preview

The LSA summaries above are **extractive** — they are actual sentences pulled directly from the original posts and comments. You may notice they can sound choppy or lack smooth transitions, since the sentences were selected independently based on their importance score rather than written as a cohesive paragraph.

In the next section, we'll apply **OpenAI's gpt-4o-mini** to generate abstractive summaries of the same posts. This will let us compare:
- **LSA (extractive):** faithful to original wording, but can be disjointed
- **GPT (abstractive):** smoother and more readable, but rephrases the original text

Both approaches have their strengths — comparing them side by side will show which works better for healthcare forum data.

## Algorithm 2: OpenAI gpt-4o-mini

**gpt-4o-mini** is a smaller, cheaper version of OpenAI's GPT-4o model. It's an **abstractive** summarization method — unlike LSA which extracts existing sentences, GPT generates **brand-new sentences** that capture the meaning of the original text in its own words.

**Why it works well for healthcare data:**
- We give it a **specific role** in the prompt: "You are a healthcare data analyst." This makes it focus on healthcare-relevant details instead of generic summarization.
- It returns **structured JSON** with three fields: a summary, an importance score (0.0–1.0), and a direction (Positive/Negative/Neutral). This gives us quantitative data we can analyze later.
- Because it understands context, it produces smoother, more readable summaries than LSA.

**Trade-offs compared to LSA:**
- **Better:** Summaries read like natural paragraphs, not choppy extracted sentences
- **Better:** Can score importance and sentiment because it understands meaning
- **Worse:** Costs money per API call (~$0.30 total for 100 posts)
- **Worse:** Requires an internet connection and an OpenAI API key
- **Worse:** Slower — each call takes ~1-2 seconds vs. LSA's milliseconds

By comparing LSA (extractive, free, local) with GPT (abstractive, paid, cloud), we can see which approach works better for summarizing healthcare IT forum posts.

In [5]:
from openai import OpenAI
from dotenv import load_dotenv
import os, json, time
from tqdm import tqdm

# --- Load the API key from the .env file ---
# The .env file is at the project root, but this notebook runs from notebooks/
# so we need to go up one or two levels to find it
load_dotenv('../../.env')

# If the key wasn't found with that path, try one level up
if os.getenv('OPENAI_API_KEY') is None:
    load_dotenv('../.env')

# Check if the API key was loaded successfully
api_key = os.getenv('OPENAI_API_KEY')

if api_key:
    print(f"API key found (starts with {api_key[:8]}...)")
    # Create the OpenAI client using the API key
    client = OpenAI(api_key=api_key)
    print("OpenAI client ready")
else:
    print("WARNING: API key not found — check your .env file")

API key found (starts with sk-proj-...)
OpenAI client ready


In [6]:
def analyze_with_gpt(text, title):
    """
    Send a healthcare post to gpt-4o-mini and get back a structured JSON
    with a summary, importance score, and direction (sentiment).
    """
    try:
        # The system prompt tells GPT to act as a healthcare analyst
        # and return structured JSON — this makes the output consistent
        system_prompt = (
            "You are a healthcare data analyst reviewing posts from healthcare "
            "IT professionals. Analyze the post and return ONLY a JSON object "
            "with exactly these fields:\n"
            "- summary: 2-3 sentence summary highlighting key healthcare IT insights\n"
            "- importance_score: float between 0.0 and 1.0 indicating importance "
            "to healthcare professionals (1.0 = critical issue, 0.0 = not relevant)\n"
            "- direction: exactly one of Positive, Negative, or Neutral\n"
            "Return ONLY the JSON object, no other text."
        )

        # Send the post title and content to GPT
        # We trim content to 1500 chars to stay within token limits and save money
        response = client.chat.completions.create(
            model="gpt-4o-mini",
            max_tokens=300,
            messages=[
                {"role": "system", "content": system_prompt},
                {"role": "user", "content": f"Title: {title}\n\nPost: {text[:1500]}"}
            ]
        )

        # Get the raw text response from GPT
        response_text = response.choices[0].message.content

        # GPT sometimes wraps JSON in markdown backticks like ```json ... ```
        # We need to strip those before parsing
        response_text = response_text.strip().strip('`')
        if response_text.startswith('json'):
            response_text = response_text[4:]

        # Parse the cleaned response as JSON
        result = json.loads(response_text)

        # Validate importance_score — must be a float between 0.0 and 1.0
        score = float(result.get('importance_score', 0.5))
        if score < 0.0 or score > 1.0:
            score = 0.5
        result['importance_score'] = score

        # Validate direction — must be one of the three allowed values
        valid_directions = ['Positive', 'Negative', 'Neutral']
        if result.get('direction') not in valid_directions:
            result['direction'] = 'Neutral'

        return result

    except Exception as e:
        # If anything goes wrong (API error, bad JSON, etc.), return safe defaults
        print(f"  GPT error for '{title[:50]}': {e}")
        return {
            'summary': text[:200],
            'importance_score': 0.5,
            'direction': 'Neutral'
        }


# --- Test GPT on the first 5 posts to make sure it works ---
print("Testing GPT on first 5 posts:\n")

for i in range(5):
    title = df.iloc[i]['title']
    content = df.iloc[i]['content']

    # Run the GPT analysis
    result = analyze_with_gpt(content, title)

    print(f"Post {i+1}: {title}")
    print(f"  Summary: {result['summary'][:150]}...")
    print(f"  Score: {result['importance_score']} | Direction: {result['direction']}\n")

    # Wait between API calls to avoid hitting rate limits
    time.sleep(0.5)

print("GPT test complete")

Testing GPT on first 5 posts:

Post 1: "I want to be an Epic analyst" FAQ
  Summary: The post provides guidance for healthcare professionals looking to transition into an Epic analyst role, highlighting key strategies such as networkin...
  Score: 0.7 | Direction: Positive

Post 2: Anyone here switch careers without a degree in informatics?
  Summary: The post discusses the transition from a clinical career to Health IT without a degree in informatics, highlighting the commonality of such paths amon...
  Score: 0.7 | Direction: Positive

Post 3: I built a browser-based ambient scribe that keeps all data on the device (open source)
  Summary: The post discusses the development of a browser-based ambient scribe that operates entirely on-device, eliminating the need for backend processing or ...
  Score: 0.7 | Direction: Positive

Post 4: How to generate a list of patients in EPIC based on ICD codes for research?
  Summary: The post discusses a student's challenge in generating a patient 

In [7]:
# --- Run GPT analysis on all 100 posts ---
print("Running gpt-4o-mini on all 100 posts...\n")

# Store results in lists so we can add them to the DataFrame later
openai_summaries = []
importance_scores = []
directions = []

for i in tqdm(range(len(df)), desc="GPT Analysis"):
    title = df.iloc[i]['title']
    content = df.iloc[i]['content']

    # Run the GPT analysis for this post
    result = analyze_with_gpt(content, title)

    # Collect the results
    openai_summaries.append(result['summary'])
    importance_scores.append(result['importance_score'])
    directions.append(result['direction'])

    # Wait between API calls to avoid hitting rate limits
    time.sleep(0.5)

# Add the results as new columns in the DataFrame
df['openai_summary'] = openai_summaries
df['importance_score'] = importance_scores
df['direction'] = directions

print("\nGPT analysis complete")
print(df[['title', 'openai_summary', 'importance_score', 'direction']].head(3))

Running gpt-4o-mini on all 100 posts...



GPT Analysis: 100%|██████████| 100/100 [05:02<00:00,  3.02s/it]


GPT analysis complete
                                               title  \
0                 "I want to be an Epic analyst" FAQ   
1  Anyone here switch careers without a degree in...   
2  I built a browser-based ambient scribe that ke...   

                                      openai_summary  importance_score  \
0  The post provides insights on transitioning to...               0.7   
1  The post highlights the possibility of transit...               0.7   
2  The post discusses the development of an open-...               0.6   

  direction  
0  Positive  
1  Positive  
2   Neutral  


In [8]:
# --- Combine results and save final CSV ---

# Select only the columns we need for the final output
final_columns = ['title', 'post_url', 'lsa_summary', 'openai_summary',
                 'importance_score', 'direction']
df_results = df[final_columns]

# Check the shape — should be (100, 6)
print(f"Results shape: {df_results.shape}")
print(f"\nFirst 3 rows:")
print(df_results.head(3))

# Save to CSV in the outputs folder
df_results.to_csv('../outputs/healthcare_analysis.csv', index=False, encoding='utf-8')
print("\nSaved to outputs/healthcare_analysis.csv")

Results shape: (100, 6)

First 3 rows:
                                               title  \
0                 "I want to be an Epic analyst" FAQ   
1  Anyone here switch careers without a degree in...   
2  I built a browser-based ambient scribe that ke...   

                                            post_url  \
0  https://www.reddit.com/r/healthIT/comments/1hl...   
1  https://www.reddit.com/r/healthIT/comments/1rx...   
2  https://www.reddit.com/r/healthIT/comments/1rw...   

                                         lsa_summary  \
0  Alternatively, keep your ear out for health sy...   
1  I'm currently on the clinical side and have be...   
2  * depends on Chrome-specific features * requir...   

                                      openai_summary  importance_score  \
0  The post provides insights on transitioning to...               0.7   
1  The post highlights the possibility of transit...               0.7   
2  The post discusses the development of an open-...            

## Results Summary

Now that both algorithms have processed all 100 healthcare IT posts, here's what the data tells us:

**Direction breakdown (Positive / Negative / Neutral):**
- The direction scores from gpt-4o-mini give us a quick read on the overall tone of the healthcare IT community on Reddit. A high proportion of Neutral posts suggests informational/question-based content, while Positive/Negative splits reveal community sentiment on tools, workflows, and career paths.

**Average importance score:**
- The mean importance score across all 100 posts indicates how relevant this subreddit's content is to working healthcare IT professionals. Scores closer to 1.0 mean the community is discussing critical, actionable topics (EHR issues, compliance, interoperability). Scores closer to 0.5 suggest more casual or mixed-relevance content.

**What this tells us about the healthcare IT community:**
- r/healthIT is a mix of career advice, technical troubleshooting, and industry discussion. The GPT analysis helps us quantify what human readers already sense — some posts are urgent professional concerns, while others are casual conversations.

Full visualizations (bar charts, histograms, word clouds) are in the next section.